# Notebook 15 — Subprocess safety & zip archives

| | |
|---|---|
| **Track** | AI Engineering · `03-python-foundations/exercises/` |
| **Level** | Advanced |
| **Pattern** | *Concept* → *Runnable demo* → *Your turn* → *Solution* |

**Prerequisites:** `14-bytes-encoding-files.ipynb`

**Next up:** `16-numpy-embeddings-shape.ipynb`; then spiral / `CURRICULUM-A-Z.md`

---

## Learning objectives

- Invoke CLI tools with argv lists — avoid `shell=True` defaults.
- Capture stdout/stderr with timeouts that match ops expectations.
- Inspect `.zip` artifacts without blindly extracting trees.

## Table of contents

1. **`subprocess.run` patterns**
2. **Timeouts & error surfaces**
3. **`zipfile` introspection**
4. **Progressive drills — argv vs shell → failing commands → largest member listing**
5. **Exercise — bounded `git` revision helper**

---

## How to use this notebook

1. Run cells **top to bottom** the first time through.
2. Re-run and **change values** to test your understanding.
3. Do **Your turn** cells before opening solutions (HTML `<details>` blocks or later cells).

---


## 1 · `subprocess.run` patterns

*Explanation:* Wrap **`git`**, **`ffmpeg`**, or codegen CLIs — pass **`list[str]`** argv so shells never interpolate secrets oddly.


In [ ]:
import subprocess

completed = subprocess.run(
    ["python", "-c", "print('hello-from-cli')"],
    capture_output=True,
    text=True,
    check=True,
)
print(completed.stdout.strip())


## 2 · Timeouts & error surfaces

*Explanation:* Hanging CLI calls stall workers — **`timeout=`** converts hangs into actionable exceptions.


In [ ]:
import subprocess

try:
    subprocess.run(["python", "-c", "import time; time.sleep(10)"], timeout=0.05)
except subprocess.TimeoutExpired as exc:
    print("stopped:", type(exc).__name__)


## 3 · `zipfile` introspection

*Explanation:* Dataset bundles arrive zipped — **`ZipFile.infolist`** exposes sizes before extraction.


In [ ]:
from pathlib import Path
import zipfile

bundle = Path("_demo_bundle.zip")
with zipfile.ZipFile(bundle, "w") as zf:
    zf.writestr("small.txt", "x")
    zf.writestr("folder/big.bin", "y" * 200)

try:
    with zipfile.ZipFile(bundle) as zf:
        infos = sorted(zf.infolist(), key=lambda i: i.file_size, reverse=True)
        print([(i.filename, i.file_size) for i in infos[:2]])
finally:
    bundle.unlink(missing_ok=True)


---

## Progressive drills — **easy → harder**

**Glue code** orchestrates binaries — drills reinforce safety habits before wiring agents to local tools.


### A · Easiest — argv list quoting

Python receives arguments cleanly — no manual escaping.


In [ ]:
import subprocess

out = subprocess.run(
    ["python", "-c", "import sys; print(sys.argv[1])", "multi word argument"],
    capture_output=True,
    text=True,
    check=True,
)
print(out.stdout.strip())


### B · Medium — tolerate nonzero exits

Sometimes stderr carries signal — inspect **`returncode`** instead of crashing silently.


In [ ]:
import subprocess

completed = subprocess.run(["python", "-c", "raise SystemExit(2)"], capture_output=True)
print("code", completed.returncode)


### C · Harder — zip-slip mindset

Normalize-member paths mentally — refuse `..` escapes before extracting.


In [ ]:
from pathlib import PurePosixPath


def safe_member(name: str) -> bool:
    dest = PurePosixPath(name)
    return dest.parts and not dest.is_absolute() and ".." not in dest.parts


print(safe_member("../evil.txt"), safe_member("data/file.txt"))


### Exercise — `git_head`

Implement `git_head(repo: Path | None = None, timeout: float = 2.0) -> str` running `git rev-parse HEAD` via **`subprocess.run`**. If `repo` is given, pass **`cwd=str(repo)`**. On failure (**nonzero exit** or **`TimeoutExpired`**), return `"unknown"`. Trim whitespace from stdout.


In [ ]:
from pathlib import Path
import subprocess


def git_head(repo: Path | None = None, timeout: float = 2.0) -> str:
    raise NotImplementedError


print(git_head())


### Solution — git_head

<details>
<summary>Click to expand</summary>

```python
from pathlib import Path
import subprocess

def git_head(repo: Path | None = None, timeout: float = 2.0) -> str:
    cmd = ["git", "rev-parse", "HEAD"]
    kw = {"cwd": str(repo)} if repo else {}
    try:
        proc = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=timeout,
            check=False,
            **kw,
        )
    except subprocess.TimeoutExpired:
        return "unknown"
    if proc.returncode != 0:
        return "unknown"
    return proc.stdout.strip()
```

</details>
